# Process Mining & Time Series Pipeline

In [1]:
# Import from library
import logging

# Import from external libraries
import pandas as pd

# Import from our package
from spi_time_series import PipelineBuilder
from spi_time_series.data import Dataset
from spi_time_series.data.schemas import EventLog, PreprocessedData, RawData
from spi_time_series.preprocessing.preprocess import (
    _build_prefixes,
    clean_data,
    preprocess,
    sliding_window_factory,
    split_data,
)

# Set up logging
logging.basicConfig(level=logging.INFO, force=True)

## The Production Pipeline
Define functions required by the `PipelineBuilder` and executes the entire workflow from start to finish.


In [ ]:
def my_preprocessor(raw: RawData) -> EventLog:
    """Clean the raw event log."""
    cleaned = clean_data(raw)
    return cleaned.event_log


def my_splitter(log: EventLog) -> PreprocessedData:
    """Split the cleaned log into train/test sets."""
    train_df, test_df = split_data(log)

    def target_func(prefix: pd.DataFrame, trace: pd.DataFrame) -> int:
        return len(prefix)

    prefix_gen = sliding_window_factory(min_length=3)

    return PreprocessedData(
        train_log=_build_prefixes(train_df, prefix_gen, target_func),
        test_log=_build_prefixes(test_df, prefix_gen, target_func),
        activity_col="concept:name",
        timestamp_col="time:timestamp",
    )


# Build the pipeline.
# As of now it only includes the dataset, preprocessor, and splitter, we will add feature extractors, models, and evaluators as needed.
dataset = Dataset()


def dummy_extractor(data):
    """bybass the feature extractor"""
    pass


pipeline = (
    PipelineBuilder()
    .with_dataset(dataset)
    .with_preprocessor(my_preprocessor)
    .with_splitter(my_splitter)
    .with_feature_extractor(dummy_extractor)
    .build()
)
# Run the pipeline.
# evaluation = pipeline.run()

INFO:spi_time_series.data.dataset:Dataset found at /Users/sakkakis/Library/CloudStorage/OneDrive-StudentsRWTHAachenUniversity/Masters/Summer Senester 26/Supervised Process Intelligence/spi-time-series/data/raw/BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
/Users/sakkakis/Library/CloudStorage/OneDrive-StudentsRWTHAachenUniversity/Masters/Summer Senester 26/Supervised Process Intelligence/spi-time-series/.venv/lib/python3.14/site-packages/pm4py/utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


## Development & Debugging

Run these cells individually to dev and debug the outputs of each pipeline stage

## Data loading

In [3]:
# Quick check to see if the data is loaded correctly
dataset_dev = Dataset()
raw_dev = RawData(event_log=dataset_dev.log)

# Quick check to see if the data is loaded correctly
print(
    f"Loaded {len(raw_dev.event_log)} events with {raw_dev.event_log['case:concept:name'].nunique()} unique cases."
)


INFO:spi_time_series.data.dataset:Dataset found at /Users/sakkakis/Library/CloudStorage/OneDrive-StudentsRWTHAachenUniversity/Masters/Summer Senester 26/Supervised Process Intelligence/spi-time-series/data/raw/BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


Loaded 1202267 events with 31509 unique cases.


###  Preprocessing
Preprocessing does 3 steps:
1) Data Cleaning 
2) Data Splitting into train and test 
3) Prefix Generation + Target computation

#### Data Cleaning


In [ ]:
cleaned_dev = clean_data(raw_dev)

original_cases = raw_dev.event_log["case:concept:name"].nunique()
cleaned_cases = cleaned_dev.event_log["case:concept:name"].nunique()

print(f"Events remaining: {len(cleaned_dev.event_log):,}")
print(
    f"Cases remaining:  {cleaned_cases:,} (Retained {cleaned_cases / original_cases:.1%})"
)

INFO:spi_time_series.preprocessing.preprocess:Filtering to terminal states: ['W_Validate application', 'W_Call after offers', 'W_Call incomplete files', 'O_Cancelled', 'A_Denied']
INFO:spi_time_series.preprocessing.preprocess:Remaining events: 1193604


Events remaining: 1,193,604
Cases remaining:  31,232 (Retained 99.1%)


### Data Splitting


In [7]:
train_df, test_df = split_data(cleaned_dev.event_log)

n_train = train_df["case:concept:name"].nunique()
n_test = test_df["case:concept:name"].nunique()
total_cases = n_train + n_test

print(f"Train cases: {n_train:,} ({n_train / total_cases:.1%})")
print(f"Test cases:  {n_test:,} ({n_test / total_cases:.1%})\n")

train_end = train_df["time:timestamp"].max()
test_start = test_df["time:timestamp"].min()

if train_end <= test_start:
    print("✅ No leakage.")
else:
    print("❌ Leakage detected!")

INFO:spi_time_series.preprocessing.preprocess:Splitting data at cutoff time: 2016-10-26 23:12:57.345400064+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. Cutoff: 2016-10-26 23:12:57.345400064+00:00
INFO:spi_time_series.preprocessing.preprocess:Train: 23837 cases, Test: 5400 cases.


Train cases: 23,837 (81.5%)
Test cases:  5,400 (18.5%)

✅ No leakage.


### Prefix generation
Prefix are created and returned as generators.

In [8]:
small_cases = train_df["case:concept:name"].unique()[:5]
df_small = train_df[train_df["case:concept:name"].isin(small_cases)]

prefix_generator = sliding_window_factory(min_length=3, max_length=None)


def target_func_dev(prefix: pd.DataFrame, trace: pd.DataFrame) -> int:
    return len(prefix)


dev_prefixes = _build_prefixes(df_small, prefix_generator, target_func_dev)

prefix_sample = next(dev_prefixes)

print(f"Case ID: {prefix_sample.case_id}")
print(f"Target Value: {prefix_sample.target}")
print(prefix_sample.prefix.head())

Case ID: Application_1000086665
Target Value: 22
   EventOrigin lifecycle:transition  OfferedAmount case:ApplicationType  \
0  Application             complete            NaN           New credit   
1  Application             complete            NaN           New credit   
2     Workflow             schedule            NaN           New credit   

           concept:name           case:LoanGoal OfferID  \
0  A_Create Application  Other, see explanation    None   
1           A_Submitted  Other, see explanation    None   
2        W_Handle leads  Other, see explanation    None   

                  EventID Selected Accepted  ...       Action  \
0  Application_1000086665     None     None  ...      Created   
1     ApplState_161925113     None     None  ...  statechange   
2      Workitem_747707399     None     None  ...      Created   

  case:RequestedAmount  CreditScore  MonthlyCost  \
0               5000.0          NaN          NaN   
1               5000.0          NaN          NaN